# MiniCoder Colab: Tiny 학습·Drive 체크포인트·재개·내보내기

먼저 런타임 유형에서 GPU를 직접 선택하세요. 무료 GPU는 계정별·시점별로 제공되지 않을 수 있고, session quota/종료시각은 자동 조회하거나 보장하지 않습니다. 기본 데이터 경로는 합성 smoke data입니다. 공개/사용자 데이터는 라이선스와 출처를 명시해야 합니다.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile, shutil
drive.mount('/content/drive')
uploaded = files.upload()  # upload the delivered MiniCoder ZIP
zip_path = next(Path(name) for name in uploaded if name.lower().endswith('.zip'))
project_root = Path('/content/MiniCoder')
shutil.rmtree(project_root, ignore_errors=True); project_root.mkdir(parents=True)
with zipfile.ZipFile(zip_path) as zf:
    root = project_root.resolve()
    for info in zf.infolist():
        target = (project_root / info.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError(f'Unsafe ZIP member: {info.filename}')
    zf.extractall(project_root)
candidates = [p for p in [project_root, *project_root.iterdir()] if (p/'model'/'config.py').exists()]
assert candidates, 'ZIP must contain model/config.py'
PROJECT = candidates[0]
print('Project:', PROJECT)
%cd $PROJECT

In [ ]:
%pip install -r requirements-training.txt
# For DATASET_MODE="public" only, run: %pip install -r requirements-public.txt
import torch
print('PyTorch:', torch.__version__, 'CUDA runtime:', torch.version.cuda, 'GPU assigned:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), 'VRAM total/free:', torch.cuda.get_device_properties(0).total_memory, torch.cuda.mem_get_info(0)[0])
else:
    print('CPU only. Provider quota is unknown and no GPU is claimed.')

In [ ]:
# Choose synthetic for the smoke path, public for an explicitly licensed Hugging Face dataset,
# or local for a dataset file/directory you own or are licensed to use.
DATASET_MODE = "synthetic"  # synthetic | public | local
SPLIT_OUT = "train"

from pathlib import Path
import json
from dataset.pipeline import DataConfig, build, tokenize_split, update_manifest_token_counts
from dataset.tokenizer import MiniCoderTokenizer
from dataset.registry import DatasetRegistry
from dataset.synthetic import write_smoke_corpus

raw = PROJECT / "raw_dataset"
data = PROJECT / "processed_dataset"
registry = DatasetRegistry(PROJECT / "data" / "dataset_registry.json")
if DATASET_MODE == "synthetic":
    write_smoke_corpus(str(raw), n=160)
    sources = [
        {"path": str(raw / "code"), "name": "synthetic_code", "license": "CC0-1.0"},
        {"path": str(raw / "pairs.jsonl"), "name": "synthetic_pairs", "license": "CC0-1.0"},
    ]
elif DATASET_MODE == "public":
    # Install the optional dependency first: %pip install -r requirements-public.txt
    # Verify the upstream license and official dataset page yourself; values below are examples to replace.
    from dataset.public import load_public_hf_dataset
    DATASET_ID = ""       # set after checking the official dataset card
    DATASET_LICENSE = ""   # set the actual license; no default license is assumed
    DATASET_URL = ""       # official HTTPS dataset card/source page
    assert DATASET_ID and DATASET_LICENSE and DATASET_URL, "Fill verified public dataset metadata first"
    fetched = load_public_hf_dataset(DATASET_ID, raw / "public.jsonl", license_id=DATASET_LICENSE,
                                    source_url=DATASET_URL, split=SPLIT_OUT, text_field="content",
                                    max_samples=20000)
    registry.register("public_source", raw / "public.jsonl", DATASET_LICENSE, DATASET_URL)
    sources = [{"path": str(raw / "public.jsonl"), "name": DATASET_ID, "license": DATASET_LICENSE,
                "source_url": DATASET_URL}]
    print("Public dataset provenance:", json.dumps(fetched, indent=2))
elif DATASET_MODE == "local":
    USER_DATA_PATH = Path("/path/to/your/authorized/dataset")
    USER_DATA_LICENSE = "MIT"  # use the actual license; unknown/non-reviewed licenses are blocked
    entry = registry.register("my_local_dataset", USER_DATA_PATH, USER_DATA_LICENSE,
                              source_url=None, notes="User-supplied dataset")
    sources = [{"path": entry["path"], "name": entry["name"], "license": entry["license"]}]
else:
    raise ValueError("DATASET_MODE must be synthetic, public, or local")

manifest = build(DataConfig(sources=sources, out_dir=str(data), val_ratio=0.1, min_val_samples=5))
texts = [json.loads(line)["text"] for line in (data / "train.jsonl").read_text(encoding="utf-8").splitlines()]
tokenizer = MiniCoderTokenizer(vocab_size=1000)
tokenizer.train_from_iterator(texts)
tokenizer.save(str(PROJECT / "tokenizer.json"))
split_meta = {}
for split in ("train", "val"):
    split_meta[split] = tokenize_split(data / f"{split}.jsonl", tokenizer,
                                       data / f"{split}.bin", manifest["dataset_version"])
manifest = update_manifest_token_counts(data, split_meta)
print("Dataset", manifest["dataset_version"], "tokens", manifest["tokenization"]["total_tokens"],
      "registered sources", [x["name"] for x in registry.list()])


In [ ]:
from training.config import TrainConfig
from training.trainer import build_from_dirs
from training.recovery import list_valid_checkpoints
from model.config import load_config
import json
CHECKPOINT_DIR = Path('/content/drive/MyDrive/MiniCoder/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
run_dir = PROJECT / 'runs' / 'colab_tiny'
valid = list_valid_checkpoints(CHECKPOINT_DIR)
next_target = (valid[0]['step'] + 2) if valid else 8
tcfg = TrainConfig(data_dir=str(data), run_dir=str(run_dir), checkpoint_dir=str(CHECKPOINT_DIR),
                   seq_len=64, micro_batch_size=2, grad_accum=2, max_steps=next_target,
                   warmup_steps=2, eval_every=4, eval_batches=4, save_every=2, log_every=2,
                   learning_rate=0.003, device='auto', resume=bool(valid))
result = build_from_dirs(load_config('config/model_tiny_10m.yaml'), tcfg).train()
print(json.dumps(result, indent=2))
assert result['final_step'] == next_target and result['final_eval'] is not None
print('Drive checkpoints:', [x['path'] for x in list_valid_checkpoints(CHECKPOINT_DIR)])

In [ ]:
# Simulate a later runtime: Drive is the persistent source; the next call resumes latest verified state.
from training.recovery import list_valid_checkpoints
from training.config import TrainConfig
from training.trainer import build_from_dirs
from model.config import load_config
prior = list_valid_checkpoints(CHECKPOINT_DIR)
assert prior, 'No valid checkpoint in Drive'
target = prior[0]['step'] + 2
resume_cfg = TrainConfig(**{**tcfg.__dict__, 'max_steps': target, 'resume': True})
resumed = build_from_dirs(load_config('config/model_tiny_10m.yaml'), resume_cfg).train()
assert resumed['final_step'] == target and resumed['tokens_seen'] > result['tokens_seen']
from model.export_cli import export_checkpoint
latest = list_valid_checkpoints(CHECKPOINT_DIR)[0]
export_result = export_checkpoint(latest['path'], str(PROJECT / 'output' / 'MiniCoder-Tiny'))
shutil.copy2(run_dir / 'metrics.jsonl', CHECKPOINT_DIR / 'colab_metrics.jsonl')
print('Resume:', json.dumps(resumed, indent=2))
print('Model export:', export_result)
print('To resume in a later Colab, mount the same Drive folder and rerun this notebook.')

## Limits

The notebook runs inside the Colab session that the user started; it cannot launch another Colab runtime. Checkpoints are written to Drive and validated by SHA-256 before resume. Hard disconnects can still lose updates since the last save interval. This is a tiny synthetic-data smoke/integration path, not 7B pretraining. Public data loading requires checking upstream license/network policy and may require internet to be enabled.